In [1]:
import json, re

INPUT  = r"D:\development\2026_python\2026_thecall\website7\nitearticles.json"
OUTPUT = r"D:\development\2026_python\2026_thecall\website7\nitearticles_fixed.json"

# ── Diagnose ──────────────────────────────────────────────────────────────────
raw = open(INPUT, encoding='utf-8').read()
print(f"File size: {len(raw):,} chars")

# Count the double-escaped quote sequences as a quick indicator
count = raw.count('\\\\"')
print(f"Double-escaped quote sequences found: {count}")
if count > 0:
    print("Diagnosis: double-serialization confirmed.")

# ── Fix option 1: parse with relaxed handling ─────────────────────────────────
# If the outer JSON is valid but inner strings are double-serialized,
# we can walk the records and re-parse affected fields.

try:
    data = json.loads(raw)
    print(f"Outer JSON parsed OK — {len(data)} records.")

    STRING_FIELDS = ('text_html', 'text_plain')
    fixed = 0
    for record in data:
        for field in STRING_FIELDS:
            val = record.get(field)
            # A double-serialized string will itself be a JSON-encoded string
            # (starts and ends with a quote when you look at the raw value)
            if isinstance(val, str) and val.startswith('"') and val.endswith('"'):
                try:
                    record[field] = json.loads(val)
                    fixed += 1
                except json.JSONDecodeError:
                    pass

    print(f"Fixed {fixed} double-serialized field values.")
    with open(OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Written → {OUTPUT}")

except json.JSONDecodeError as e:
    # ── Fix option 2: the outer JSON itself is broken ─────────────────────────
    # Replace \" with a placeholder, fix \\", restore — then re-parse
    print(f"Outer JSON broken at: {e}")
    print("Attempting raw string repair...")

    # Replace all \\" with a safe placeholder, then fix remaining \"
    repaired = raw.replace('\\\\"', '\\"')
    try:
        data = json.loads(repaired)
        print(f"Repair successful — {len(data)} records.")
        with open(OUTPUT, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Written → {OUTPUT}")
    except json.JSONDecodeError as e2:
        print(f"Repair failed: {e2}")
        print("The data may need to be re-exported from the database.")

File size: 1,444,296 chars
Double-escaped quote sequences found: 10458
Diagnosis: double-serialization confirmed.
Outer JSON broken at: Expecting ',' delimiter: line 1 column 6080 (char 6079)
Attempting raw string repair...
Repair successful — 137 records.
Written → D:\development\2026_python\2026_thecall\website7\nitearticles_fixed.json
